## A map, not a maximum

The only campaign here that answers *how many different ways does this go wrong* rather
than *how badly*. Its archive is keyed on the failure mode and on how close the crossing
came, so a cell that collides and a cell that gives up are different entries rather than
two points with the same score.

**What to look for:** how many rows have anything in them. An empty row is a kind of
failure this system does not exhibit — which is a result.


In [ ]:
# DATA_DIR is replaced by the service with the node being viewed. The assignment must stay
# a plain literal for that substitution to work.
DATA_DIR = ''

import json
import pandas as pd
import matplotlib.pyplot as plt

from robovast.common.analysis import CampaignDataError, open_campaign_store

def load_units(data_dir):
    """One row per evaluated cell: its parameters, objectives and measures.

    Read from the campaign's own store rather than the results index because that is where a
    SEARCH records what it scored -- the index holds per-run tables, and a search's unit of
    analysis is the cell. The store is also written as the search runs, so this works on a
    campaign that is still going or was never postprocessed.
    """
    # open_campaign_store rather than a sqlite3.connect on a path built here: it resolves the
    # campaign ROOT from data_dir, so this cell also works at a configuration node instead of
    # only at the campaign, and it is the one place that knows where the store lives.
    try:
        conn = open_campaign_store(data_dir)
    except CampaignDataError as exc:
        # Reported, not swallowed. "This campaign scored nothing" and "its record is not
        # here" are different answers and only the first is a result -- an empty frame
        # returned quietly reads as the first while meaning the second.
        print(f'[no data] {exc}')
        return pd.DataFrame()
    try:
        units = pd.read_sql_query(
            "SELECT u.paramset_id, u.config_name, u.params_json, u.objectives_json,"
            "       u.measures_json, u.n_samples, u.status, b.idx AS batch"
            "  FROM unit u LEFT JOIN batch b ON b.id = u.batch_id"
            " ORDER BY b.idx, u.id", conn)
    finally:
        conn.close()
    if units.empty:
        return units
    for col, prefix in (('params_json', ''), ('objectives_json', ''), ('measures_json', 'm_')):
        expanded = units[col].apply(lambda s: json.loads(s) if s else {}).apply(pd.Series)
        expanded.columns = [f'{prefix}{c}' for c in expanded.columns]
        units = pd.concat([units.drop(columns=[col]), expanded], axis=1)
    return units

units = load_units(DATA_DIR)
scored = units[units['status'] == 'evaluated'] if 'status' in units else units
print(f"{len(units)} cell(s) recorded, {len(scored)} scored")

if scored.empty:
    print("No scored cells yet. A search records a cell once its batch has been evaluated;"
          "\nif this campaign failed early, its controller log says why.")


In [ ]:
TITLE = 'Quality-diversity — how many kinds of trouble'

# The archive as it is keyed: which KIND of trouble, against how close it came. A filled
# square is a kind that actually happens; an empty one is a kind that does not.
if not scored.empty and 'm_failure_mode' in scored:
    modes = ['none', 'collision', 'timeout', 'goal_miss']
    fig, ax = plt.subplots(figsize=(6.5, 4))
    for i, mode in enumerate(modes):
        rows = scored[scored['m_failure_mode'] == mode]
        ax.scatter(rows['m_min_clearance'], [i] * len(rows),
                   s=110, alpha=0.75, edgecolor='black', linewidth=0.4)
    ax.set_yticks(range(len(modes))); ax.set_yticklabels(modes)
    ax.set_xlabel('minimum clearance [m]'); ax.set_ylabel('failure mode')
    ax.axvline(0, color='crimson', linestyle='--', linewidth=1, label='contact')
    ax.set_title('%s: %d distinct kind(s) observed'
                 % (TITLE, scored['m_failure_mode'].nunique()))
    ax.legend(); plt.tight_layout(); plt.show()


In [ ]:
if not scored.empty:
    failed = (scored['robustness'] < 0).sum()
    print(f"cells scored          : {len(scored)}")
    print(f"runs spent            : {int(scored['n_samples'].sum())}")
    print(f"cells that failed     : {failed}  ({failed / len(scored):.0%})")
    print(f"worst robustness      : {scored['robustness'].min():.3f}")
    print()
    modes = scored['m_failure_mode'].value_counts() if 'm_failure_mode' in scored else {}
    print("What this campaign is FOR -- how many KINDS of trouble exist:")
    print(f"  distinct failure modes found : {len(modes)}")
    for mode, count in modes.items():
        sub = scored[scored['m_failure_mode'] == mode]['robustness']
        print(f"    {mode:<12} {count:>3} cell(s), robustness {sub.min():.3f} .. {sub.max():.3f}")
    print("  An archive is judged on the SPREAD it fills, not on its best cell: two modes")
    print("  found once each beat one mode found forty times.")
